# Blocco B — Costruzione del grafo geopolitico-cyber

**Nodi** = paesi: i 17 del progetto (con i loro 28 profili trimestrali arricchiti = qualitativo dal Blocco A + numerico dai CSV) e i paesi *periferici* esterni che compaiono nelle relazioni.

**Archi** diretti e datati, tutti da dati strutturati (zero LLM):
- **cyber** — attaccante → vittima (CFR)
- **migrazione** — origine → destinazione (UNHCR)
- **coinvolgimento militare** — interventore → teatro (ACLED, forze militari straniere)

La logica di costruzione sta nei moduli `src/graph/build_edges.py` e `build_graph.py`; qui li orchestriamo, controlliamo il risultato e lo visualizziamo.

In [ ]:
import sys
from pathlib import Path
RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE))

from src.graph.build_graph import costruisci_grafo, riepilogo, salva

G = costruisci_grafo()
riepilogo(G)
salva(G)   # -> data/processed/graphs/grafo.pickle + archi.csv

## Controlli di sanità — le relazioni principali

In [ ]:
import pandas as pd
archi = pd.read_csv(RADICE / "data/processed/graphs/archi.csv")

print("TOP attaccanti cyber (n. archi uscenti):")
print(archi[archi.tipo == "cyber"].da.value_counts().head(6), "\n")

print("TOP coinvolgimenti militari (eventi):")
print(archi[archi.tipo == "coinvolgimento_militare"].groupby(["da", "a"]).peso.sum().sort_values(ascending=False).head(6), "\n")

print("TOP flussi migratori (rifugiati):")
print(archi[archi.tipo == "migrazione"].groupby(["da", "a"]).peso.sum().sort_values(ascending=False).head(6))

## Visualizzazione su mappa

Mappa tematica dei 17 paesi del progetto. Si apre **vuota**: si vedono solo i nostri paesi, evidenziati e colorati per gruppo, su un planisfero di contesto. Dal pannello si accende **un tipo di relazione alla volta** (🔴 Cyber · 🔵 Migrazione · 🟠 Coinvolgimento militare).

Ogni arco è **diretto** — la freccia indica il verso (attaccante→vittima, origine→destinazione, interventore→teatro) — e passandoci sopra mostra verso e intensità. **Rotella** = zoom · **trascina** = sposta · **clic su un paese** = isola le sue relazioni.

Solo relazioni **tra i 17 paesi** (niente periferici, per leggibilità). Tutto è generato da `src/graph/build_map.py` in un unico file HTML autosufficiente (`grafo_mappa.html`), senza librerie esterne.

In [ ]:
from src.graph.build_map import costruisci
from IPython.display import IFrame

costruisci()   # -> data/processed/graphs/grafo_mappa.html (HTML autosufficiente)
IFrame(src="../data/processed/graphs/grafo_mappa.html", width="100%", height=760)

## Grafo completo (per l'analisi)

La mappa qui sopra mostra volutamente solo le relazioni **tra i 17 paesi**. Il grafo **completo** — 172 nodi, inclusi i paesi *periferici* (vittime cyber, teatri e destinazioni migratorie esterni) e tutti i ~7.300 archi — resta salvato in `grafo.pickle` e in `archi.csv` per le analisi successive (Blocco C/D). La visualizzazione dei periferici sulla mappa è rimandata a quando servirà.